# AI-Assisted Coding and Analysis of Expert Interviews
## Deep Mediatization and AI Transformation of Performing and Screen Arts

This notebook supports reproducible qualitative analysis of 11 expert interview transcripts in `.txt` format, contributing to an academic article written in English.

The workflow combines (1) interview structure and interviewer/respondent role identification, (2) deductive and inductive thematic coding, (3) verbatim-source evidence verification, (4) within-case and cross-case synthesis, and (5) thematic alignment with a separate mass survey.

**Language policy:** All analytical text, code comments, codebook definitions, generated memos, and manuscript-ready prose must be written in **academic English**. Original Ukrainian or Russian interview excerpts must remain verbatim, with no fabricated English wording. When English quotations are needed for publication, translate verified originals separately, retaining the original text and a translation audit trail.

**Evidence policy:** Interviews support attributed qualitative interpretations, not population prevalence estimates. Preserve dissenting cases and uncertainty caused by automatic speech recognition (ASR) errors. No external claims may be presented as if they were interview evidence.


In [1]:

# If the packages are missing, uncomment and run once:
# %pip install -q -U openai pandas openpyxl

from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from difflib import SequenceMatcher
from getpass import getpass
from pathlib import Path
from typing import Any

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None


## 1. Configuration

In [ ]:

# Directory containing TXT interview transcripts.
INPUT_DIR = Path("/Mediatization_KII")

# Optional directory of experts, formatted like KII_list.txt.
KII_LIST_PATH: Path | None = Path("/Mediatization_KII/KII_list.txt")

# Optional interview guide: if missing or None,
# use the built-in interview question map below.
INTERVIEW_GUIDE_PATH: Path | None = Path("/Mediatization_KII/Manual.txt")

# The survey workbook is used only to align interview themes with the future
# quantitative component. Email addresses and timestamps are NOT sent to the model.
SURVEY_XLSX: Path | None = Path("/Mediatization_KII/Медіатизація культури.xlsx")
SURVEY_SHEET = "Form Responses 1"

OUTPUT_DIR = Path("/Mediatization_KII_output")
RESULT_XLSX = OUTPUT_DIR / "mediatization_kii_analysis.xlsx"
ARTICLE_MEMO_MD = OUTPUT_DIR / "qualitative_article_memo.md"
CACHE_DIR = OUTPUT_DIR / "_cache"

# Cost–quality settings; change model IDs according to your API project availability.
CODING_MODEL = "gpt-5.6-luna"
SYNTHESIS_MODEL = "gpt-5.6-sol"

OUTPUT_LANGUAGE = "English"
API_KEY_PATH: Path | None = Path("/keys/api_openai.txt")

REQUEST_TIMEOUT_SECONDS = 300
MAX_RETRIES = 4
RETRY_BASE_SECONDS = 4
SLEEP_BETWEEN_REQUESTS = 0.4

# Chunk size: split preferentially at speaker-turn and Q&A boundaries.
CHUNK_MAX_CHARS = 38_000
CHUNK_OVERLAP_TURNS = 1
MAX_SYNTHESIS_INPUT_CHARS = 180_000

SKIP_CACHED = True
RUN_ANALYSIS = True   # Inspect the preview first, then change to True.

SUPPORTED_EXTENSIONS = {".txt"}
IGNORED_NAMES = {"kii_list.txt", "manual.txt", "instruction.txt"}


## 2. Research framework and deductive codebook

The predefined codes are **sensitizing concepts**, not predetermined findings. The model must also return an `emergent_code` whenever an analytically significant issue falls outside the codebook.

The central distinction is between ordinary digitalization, deep mediatization through platform and visibility logics, and AI-driven transformations of creative production and professional boundaries.


In [3]:

ARTICLE_TITLE = "Deep Mediatization and AI Transformation of Performing and Screen Arts"

RESEARCH_QUESTIONS = {
    "RQ1": 'How have digital media become a professional infrastructure in music, theatre, and film?',
    "RQ2": 'How do platformization, algorithmic visibility, and metrics shape professional practices and artistic decisions?',
    "RQ3": 'How is AI used in creative production, and where do experts draw the line between augmentation and substitution of human labour?',
    "RQ4": 'What ethical, legal, and professional risks of AI do creative-industry practitioners identify?',
    "RQ5": 'How does mediatization transform audience relations, the economics of creative work, and the form of cultural products?',
    "RQ6": 'How do music, theatre, and film differ in the depth and configuration of mediatization and AI transformation?',
}

CODEBOOK = {
    "DIGITAL_INFRASTRUCTURE": 'Digital tools, services and software as routine infrastructure of creative work.',
    "PRODUCTION_ACCELERATION": 'Acceleration, simplification, or automation of production processes.',
    "AI_AUGMENTATION": 'AI as an assistive tool for research, analysis, ideation, technical work, generation of components, programming, or related tasks.',
    "AI_SUBSTITUTION": 'Actual or anticipated substitution of human creative or professional work by AI.',
    "AUTHORSHIP_IP": 'Authorship, intellectual property, voice and likeness rights, training data, and unauthorized reuse.',
    "AI_ETHICS_QUALITY": 'Ethical concerns, professional standards, quality, originality, and dependence on AI.',
    "PLATFORMIZATION": 'Reliance on digital platforms, streaming services, social networks, and their infrastructure.',
    "ALGORITHMIC_VISIBILITY": 'Algorithmic promotion, reach, recommendations, and the pressure to maintain visibility.',
    "METRICS_DATAFICATION": 'Views, followers, reach, charts, ratings, and quantified performance indicators shaping professional practice.',
    "PERSONAL_BRAND_VISIBILITY_LABOR": 'Personal branding, recurrent content creation, self-presentation, and the labour of sustaining online visibility.',
    "DIGITAL_PRESSURE_BURNOUT": 'Pressure to remain digitally active, scarcity of time and resources, and emotional exhaustion.',
    "AUDIENCE_CONNECTIVITY": 'More direct, faster, or continuous connections between creators and audiences through digital channels.',
    "AUDIENCE_FEEDBACK_CREATIVE_INFLUENCE": 'Influence of online responses, comments, hostility, demand, or audience expectations on creative decisions.',
    "ECONOMIC_MONETIZATION": 'Monetization, royalties, fees, advertising, commissions, and the economic effects of digital platforms.',
    "PROFESSIONAL_OPPORTUNITIES_NETWORKS": 'Collaborations, auditions, networking, festivals, commissions, and professional opportunities arising online.',
    "FORM_TRANSFORMATION": 'Changes in the duration, structure, pacing, visual form, or format of creative products linked to digital consumption.',
    "MOBILE_SHORT_VIDEO_LOGIC": 'TikTok, Reels, mobile viewing, short video formats, and demands for instantaneous audience engagement.',
    "STREAMING_DISTRIBUTION": 'Streaming services and digital distribution as routes to audiences and structural forces within creative industries.',
    "LIVENESS_AUTHENTICITY_EMBODIMENT": 'Liveness, embodiment, co-presence, emotional expression, and authenticity as values or limits of technological substitution.',
    "HYBRIDIZATION_TRANSMEDIA": 'Blending of live, screen-based and digital forms, VR/AR, transmedia, interactive, and hybrid formats.',
    "GATEKEEPING_NDA_INSTITUTIONAL_CONSTRAINT": 'NDAs, major studios, producers, approvals, and other institutional constraints on media presence or artistic choices.',
    "PROFESSIONAL_IDENTITY_BOUNDARIES": 'Understandings of professional identity, competence, craft, and the boundaries of technologically substitutable work.',
    "WAR_CONTEXT_MEDIATION": 'Effects of war on communication, content decisions, appropriateness of self-promotion, and emotional conditions of creative work.',
    "FUTURE_EXPECTATIONS": 'Expectations about the future of professions, creative sectors, digital media, and AI.',
}

MEDIATIZATION_MECHANISMS = [
    "digitalization", "platformization", "algorithmization", "datafication",
    "connectivity", "visibility_logic", "audience_feedback", "monetization",
    "automation_ai", "hybridization", "institutional_constraint", "none"
]

SECTORS = ["music", "theatre", "film", "theatre_film", "mixed", "unknown"]
STANCES = ["positive", "negative", "ambivalent", "neutral", "not_applicable"]
AI_RELEVANCE = ["direct", "indirect", "none"]


## 3. Interview guide and mass-survey thematic alignment

In [4]:

# Concise machine-readable guide map. If INTERVIEW_GUIDE_PATH exists,
# its full text will also be supplied to the model as contextual material.
QUESTION_MAP = {
    "Q1": 'Professional background and career trajectory',
    "Q2": 'Creative sector and role in production',
    "Q3": 'Changes in professional practice due to digital technologies',
    "Q4": 'Extent of digital technology use',
    "Q5": 'Most important or indispensable technologies',
    "Q6": 'Changes in the creative production process',
    "Q7": 'Role of social media',
    "Q8": 'Platform-specific content creation',
    "Q9": 'Dependence of artistic success on media visibility',
    "Q10": 'Pressure of constant digital activity',
    "Q11": 'Changes in audience interaction',
    "Q12": 'Significance of digital metrics',
    "Q13": 'Influence of online audience feedback on creative decisions',
    "Q14": 'Digital platforms in professional activity',
    "Q15": 'Professional opportunities through online channels',
    "Q16": 'Economic effects of digital platforms',
    "Q17": 'Attitudes towards AI in creative work',
    "Q18": 'Ethical and professional risks of AI',
    "Q19": 'Future of the profession under mediatization',
    "Q20+": 'Sector-specific questions: theatre / film / music',
}

# These thematic axes enable later integration with the mass survey.
SURVEY_ALIGNMENT_AXES = {
    "cultural_participation_format": ["форматі", "культурн"],
    "social_media_following": ["підписані", "сторінки митців"],
    "cultural_content_preferences": ["контенті", "культурну тематику"],
    "online_expression": ["висловлювати власну думку", "онлайн-середовищ"],
    "music_discovery": ["дізнаєтеся про нових виконавців"],
    "music_visuality": ["візуальний контент", "сприйняття музики"],
    "musician_media_dependence": ["успіх сучасного музиканта"],
    "film_discovery_virality": ["вірусному контенту", "нові фільми"],
    "film_media_hype": ["шум", "суб'єктивне сприйняття"],
    "film_online_discussion": ["обговорювати фільми онлайн"],
    "film_mediatization_vs_quality": ["медіатизації", "художньої якості"],
    "theatre_discovery": ["нові вистави"],
    "theatre_social_content": ["контенту від театрів"],
    "theatre_media_image": ["імідж театру", "акторів у медіа"],
    "theatre_online_streaming": ["онлайн-трансляції вистав"],
    "theatre_social_interaction": ["взаємодію з театром"],
    "theatre_media_to_attendance": ["впливає на моє рішення піти на виставу"],
    "live_theatre_vs_online": ["живий", "конкурує з онлайн-контентом"],
}


## 4. Structured Outputs schemas

In [5]:

def enum_schema(values: list[str]) -> dict[str, Any]:
    return {"type": "string", "enum": values}

STRUCTURE_SCHEMA = {
    "type": "object",
    "properties": {
        "transcript_format": enum_schema(["speaker_labeled", "numbered_qa", "mixed", "unknown"]),
        "primary_language": enum_schema(["Ukrainian", "Russian", "English", "Mixed", "Other"]),
        "interviewer_label": {"type": "string"},
        "expert_label": {"type": "string"},
        "role_confidence": {"type": "integer", "minimum": 1, "maximum": 3},
        "role_rationale": {"type": "string"},
        "respondent_name_from_text": {"type": "string"},
        "sector": enum_schema(SECTORS),
        "quality_notes": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "transcript_format", "primary_language", "interviewer_label", "expert_label",
        "role_confidence", "role_rationale", "respondent_name_from_text", "sector", "quality_notes"
    ],
    "additionalProperties": False,
}

EVIDENCE_ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "question_ids": {"type": "array", "items": {"type": "string"}},
        "quote": {"type": "string"},
        "paraphrase": {"type": "string"},
        "primary_code": enum_schema(list(CODEBOOK)),
        "secondary_codes": {"type": "array", "items": enum_schema(list(CODEBOOK))},
        "emergent_code": {"type": "string"},
        "mediatization_mechanisms": {"type": "array", "items": enum_schema(MEDIATIZATION_MECHANISMS)},
        "stance": enum_schema(STANCES),
        "ai_relevance": enum_schema(AI_RELEVANCE),
        "sector_specificity": enum_schema(["cross_sector", "sector_specific", "unclear"]),
        "analytic_memo": {"type": "string"},
        "confidence": {"type": "integer", "minimum": 1, "maximum": 3},
    },
    "required": [
        "question_ids", "quote", "paraphrase", "primary_code", "secondary_codes",
        "emergent_code", "mediatization_mechanisms", "stance", "ai_relevance",
        "sector_specificity", "analytic_memo", "confidence"
    ],
    "additionalProperties": False,
}

CHUNK_SCHEMA = {
    "type": "object",
    "properties": {
        "chunk_summary": {"type": "string"},
        "evidence": {"type": "array", "items": EVIDENCE_ITEM_SCHEMA},
        "contradictions_or_qualifications": {"type": "array", "items": {"type": "string"}},
        "missing_or_uncertain_material": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["chunk_summary", "evidence", "contradictions_or_qualifications", "missing_or_uncertain_material"],
    "additionalProperties": False,
}

INTERVIEW_SUMMARY_SCHEMA = {
    "type": "object",
    "properties": {
        "analytical_summary": {"type": "string"},
        "mediatization_profile": {"type": "string"},
        "ai_transformation_profile": {"type": "string"},
        "professional_identity_and_boundaries": {"type": "string"},
        "audience_and_visibility": {"type": "string"},
        "economic_and_platform_dimension": {"type": "string"},
        "form_and_product_transformation": {"type": "string"},
        "main_tensions": {"type": "array", "items": {"type": "string"}},
        "sector_specific_findings": {"type": "array", "items": {"type": "string"}},
        "emergent_themes": {"type": "array", "items": {"type": "string"}},
        "article_contributions": {"type": "array", "items": {"type": "string"}},
        "possible_propositions": {"type": "array", "items": {"type": "string"}},
        "limitations": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "analytical_summary", "mediatization_profile", "ai_transformation_profile",
        "professional_identity_and_boundaries", "audience_and_visibility",
        "economic_and_platform_dimension", "form_and_product_transformation",
        "main_tensions", "sector_specific_findings", "emergent_themes",
        "article_contributions", "possible_propositions", "limitations"
    ],
    "additionalProperties": False,
}

CROSS_CASE_SCHEMA = {
    "type": "object",
    "properties": {
        "central_argument": {"type": "string"},
        "deep_mediatization_findings": {"type": "array", "items": {"type": "string"}},
        "ai_transformation_findings": {"type": "array", "items": {"type": "string"}},
        "cross_sector_commonalities": {"type": "array", "items": {"type": "string"}},
        "sector_differences": {"type": "array", "items": {"type": "string"}},
        "key_tensions": {"type": "array", "items": {"type": "string"}},
        "negative_cases_and_contradictions": {"type": "array", "items": {"type": "string"}},
        "qualitative_results_draft": {"type": "string"},
        "discussion_points": {"type": "array", "items": {"type": "string"}},
        "survey_integration_plan": {"type": "array", "items": {"type": "string"}},
        "claims_not_supported": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "central_argument", "deep_mediatization_findings", "ai_transformation_findings",
        "cross_sector_commonalities", "sector_differences", "key_tensions",
        "negative_cases_and_contradictions", "qualitative_results_draft",
        "discussion_points", "survey_integration_plan", "claims_not_supported"
    ],
    "additionalProperties": False,
}


## 5. Input loading, metadata matching, and transcript format identification

In [6]:

@dataclass
class ExpertMeta:
    name: str
    age: int | None
    description: str
    sector: str


def read_text_file(path: Path) -> tuple[str, str]:
    for enc in ("utf-8-sig", "utf-8", "cp1251", "windows-1251"):
        try:
            return path.read_text(encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raw = path.read_bytes()
    return raw.decode("utf-8", errors="replace"), "utf-8-replace"


def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def infer_sector_from_heading(heading: str) -> str:
    h = heading.lower()
    if "муз" in h:
        return "music"
    if "театр" in h:
        return "theatre"
    if "кіно" in h or "кино" in h:
        return "film"
    return "unknown"


def load_expert_directory(path: Path | None) -> list[ExpertMeta]:
    if path is None or not path.exists():
        return []
    text, _ = read_text_file(path)
    experts: list[ExpertMeta] = []
    current_sector = "unknown"
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.rstrip(":").lower() in {"музика", "театр", "кіно", "кино"}:
            current_sector = infer_sector_from_heading(line)
            continue
        m = re.match(r"^(.+?)\s*\((\d{1,3})\)\s*[-–—]\s*(.+)$", line)
        if m:
            experts.append(ExpertMeta(
                name=m.group(1).strip(),
                age=int(m.group(2)),
                description=m.group(3).strip(),
                sector=current_sector,
            ))
    return experts


def norm_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-zа-яіїєґ0-9]+", " ", s, flags=re.I)
    return " ".join(s.split())


def match_expert(filename: str, experts: list[ExpertMeta]) -> tuple[ExpertMeta | None, float]:
    stem = norm_name(Path(filename).stem.replace("_ua", "").replace("_ru", "").replace("_en", ""))
    if not experts:
        return None, 0.0
    scored = []
    for expert in experts:
        name = norm_name(expert.name)
        surname = name.split()[-1] if name.split() else name
        score = max(
            SequenceMatcher(None, stem, name).ratio(),
            SequenceMatcher(None, stem, surname).ratio(),
            1.0 if surname and surname in stem else 0.0,
        )
        scored.append((score, expert))
    score, expert = max(scored, key=lambda x: x[0])
    return (expert, score) if score >= 0.48 else (None, score)


def discover_transcripts(input_dir: Path) -> list[Path]:
    ignored = {norm_name(x) for x in IGNORED_NAMES}
    files = []
    for p in sorted(input_dir.glob("*.txt")):
        if p.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        if norm_name(p.name) in ignored:
            continue
        files.append(p)
    return files


def load_guide_text() -> str:
    if INTERVIEW_GUIDE_PATH is not None and INTERVIEW_GUIDE_PATH.exists():
        txt, _ = read_text_file(INTERVIEW_GUIDE_PATH)
        return clean_text(txt)
    return "\n".join(f"{k}: {v}" for k, v in QUESTION_MAP.items())


def load_survey_headers() -> list[str]:
    if SURVEY_XLSX is None or not SURVEY_XLSX.exists():
        return []
    wb = load_workbook(SURVEY_XLSX, read_only=False, data_only=True)
    ws = wb[SURVEY_SHEET] if SURVEY_SHEET in wb.sheetnames else wb[wb.sheetnames[0]]
    headers = [str(c.value).strip() if c.value is not None else "" for c in ws[1]]
    # Exclude personally identifiable information and administrative fields.
    excluded = {"timestamp", "email address", "email", "e-mail"}
    return [h for h in headers if h and h.lower() not in excluded]


def map_survey_axes(headers: list[str]) -> dict[str, list[str]]:
    result: dict[str, list[str]] = {}
    for axis, needles in SURVEY_ALIGNMENT_AXES.items():
        matched = [h for h in headers if all(n.lower() in h.lower() for n in needles)]
        if not matched:
            matched = [h for h in headers if any(n.lower() in h.lower() for n in needles)]
        result[axis] = matched
    return result


def resolve_sector(expert_meta: ExpertMeta | None, structure: dict[str, Any]) -> str:
    detected = structure.get("sector", "unknown")
    # Preserve explicitly mixed professional positions (e.g. theatre + film) detected from the transcript.
    if detected in {"theatre_film", "mixed"}:
        return detected
    if expert_meta is not None and expert_meta.sector != "unknown":
        return expert_meta.sector
    return detected


## 6. API calls, structured responses, caching, and logging

In [7]:

API_CALL_LOG: list[dict[str, Any]] = []


def load_api_key() -> str:
    env_key = os.getenv("OPENAI_API_KEY", "").strip()
    if env_key:
        return env_key
    if API_KEY_PATH is not None and Path(API_KEY_PATH).exists():
        value = Path(API_KEY_PATH).read_text(encoding="utf-8").strip()
        if value:
            return value
    entered = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not entered:
        raise ValueError("OPENAI_API_KEY was not provided")
    return entered


def safe_slug(value: str) -> str:
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("_")
    return value[:120] or "item"


def fingerprint(*parts: str) -> str:
    h = hashlib.sha256()
    for part in parts:
        h.update(part.encode("utf-8", errors="replace"))
        h.update(b"\0")
    return h.hexdigest()


def call_structured(
    client: OpenAI,
    *,
    model: str,
    instructions: str,
    input_text: str,
    schema: dict[str, Any],
    schema_name: str,
    max_output_tokens: int,
    stage: str,
    transcript_id: str,
) -> dict[str, Any]:
    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        started = time.time()
        try:
            response = client.responses.create(
                model=model,
                instructions=instructions,
                input=input_text,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": schema_name,
                        "strict": True,
                        "schema": schema,
                    }
                },
                max_output_tokens=max_output_tokens,
                store=False,
            )
            if not response.output_text:
                raise ValueError("Empty model response")
            parsed = json.loads(response.output_text)
            usage = getattr(response, "usage", None)
            API_CALL_LOG.append({
                "transcript_id": transcript_id,
                "stage": stage,
                "model": model,
                "attempt": attempt,
                "status": "ok",
                "elapsed_seconds": round(time.time() - started, 2),
                "input_tokens": getattr(usage, "input_tokens", None),
                "output_tokens": getattr(usage, "output_tokens", None),
                "response_id": getattr(response, "id", ""),
                "error": "",
            })
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            return parsed
        except Exception as exc:
            last_error = exc
            API_CALL_LOG.append({
                "transcript_id": transcript_id,
                "stage": stage,
                "model": model,
                "attempt": attempt,
                "status": "failed_attempt",
                "elapsed_seconds": round(time.time() - started, 2),
                "input_tokens": None,
                "output_tokens": None,
                "response_id": "",
                "error": str(exc),
            })
            if attempt < MAX_RETRIES:
                delay = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 1.25)
                print(f"API error, retry in {delay:.1f}s: {exc}")
                time.sleep(delay)
    raise RuntimeError(f"API call failed after {MAX_RETRIES} attempts: {last_error}")


def load_cache(path: Path) -> dict[str, Any] | None:
    if SKIP_CACHED and path.exists() and path.stat().st_size > 0:
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def save_cache(path: Path, data: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


## 7. Speaker-role identification: interviewer versus expert

In [8]:

COMMON_EVIDENCE_RULES = f"""
You are coding expert interviews for the article: {ARTICLE_TITLE}.
Analytical output language: {OUTPUT_LANGUAGE} (formal academic English in every analytical field).

Evidence rules:
1. Use ONLY the supplied transcript. Do not add external facts.
2. Treat every substantive statement as the respondent's account/opinion unless the transcript itself establishes otherwise.
3. Never assume Speaker A is the interviewer. Use the supplied role mapping.
4. Do not code interviewer wording as expert evidence unless the expert explicitly adopts/affirms it in their own response.
5. Preserve contradictions, refusals, uncertainty, and negative cases.
6. For every evidence item, quote must be a SHORT CONTIGUOUS VERBATIM passage copied from the transcript chunk.
   Do not clean, translate, repair, combine or silently correct the quote. Write all interpretations, code labels, summaries, and memos in English; preserve quotations in the original source language.
7. If ASR damage makes a passage uncertain, do not create a quote from it. Mention the uncertainty instead.
8. Fixed codes are sensitizing concepts, not mandatory categories. Use emergent_code for analytically important material that does not fit them.
9. Do not infer population prevalence from expert interviews.
10. Distinguish digitalization (tools/processes), platformization, algorithmization, datafication, connectivity,
    visibility labour, audience feedback, monetization, AI automation and hybridization when the text supports the distinction.
11. Distinguish AI as augmentation from AI as substitution/displacement. Do not collapse them.
12. Distinguish the technical transformation of production from transformation of artistic form/content and from transformation of professional identity.
""".strip()


def detect_structure(
    client: OpenAI,
    transcript_id: str,
    text: str,
    candidate_expert: ExpertMeta | None,
    match_score: float,
    guide_text: str,
) -> dict[str, Any]:
    key = fingerprint(text[:50_000], guide_text, candidate_expert.name if candidate_expert else "")
    cache_path = CACHE_DIR / safe_slug(transcript_id) / f"{key[:20]}_STRUCTURE.json"
    cached = load_cache(cache_path)
    if cached is not None:
        return cached

    candidate = None
    if candidate_expert:
        candidate = {
            "name": candidate_expert.name,
            "age": candidate_expert.age,
            "description": candidate_expert.description,
            "sector": candidate_expert.sector,
            "filename_match_score": round(match_score, 3),
        }

    instructions = """
Determine the structural format of an expert interview and identify interviewer vs expert roles.
The transcript may use Speaker A/B labels, but either label can be the interviewer.
A different format may list numbered questionnaire items followed by answers without speaker labels.
Use conversational evidence: the interviewer introduces blocks, asks questions and thanks/responds briefly;
the expert provides autobiographical, professional and evaluative answers.
Do not infer a real name unless the text or supplied candidate metadata supports it.
If numbered Q&A has no speaker labels, return interviewer_label='QUESTION_TEXT' and expert_label='ANSWER_TEXT'.
If role attribution is uncertain, say so and lower role_confidence.
""".strip()

    prompt = json.dumps({
        "candidate_expert_from_filename": candidate,
        "interview_guide": guide_text[:20_000],
        "transcript_sample": text[:50_000],
    }, ensure_ascii=False, indent=2)

    result = call_structured(
        client,
        model=CODING_MODEL,
        instructions=instructions,
        input_text=prompt,
        schema=STRUCTURE_SCHEMA,
        schema_name="interview_structure",
        max_output_tokens=1_200,
        stage="structure_detection",
        transcript_id=transcript_id,
    )
    save_cache(cache_path, result)
    return result


## 8. Transcript parsing and speaker-turn / Q&A chunking

In [9]:

SPEAKER_RE = re.compile(r"(?m)^\s*(Speaker\s+[A-Z]|SPEAKER_\d+|Спікер\s+[A-ZА-Я]|Спикер\s+[A-ZА-Я])\s*:\s*")
NUMBERED_Q_RE = re.compile(r"(?m)^\s*(\d{1,2})[\.\)]\s+(.+)$")


def parse_speaker_turns(text: str) -> list[dict[str, str]]:
    matches = list(SPEAKER_RE.finditer(text))
    if not matches:
        return []
    turns = []
    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        turns.append({"label": m.group(1).strip(), "text": text[start:end].strip()})
    return [t for t in turns if t["text"]]


def parse_numbered_qa(text: str) -> list[dict[str, str]]:
    matches = list(NUMBERED_Q_RE.finditer(text))
    if not matches:
        return []
    items = []
    for i, m in enumerate(matches):
        qnum = m.group(1)
        question = m.group(2).strip()
        answer_start = text.find("\n", m.end())
        if answer_start < 0:
            answer_start = m.end()
        else:
            answer_start += 1
        answer_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        answer = text[answer_start:answer_end].strip()
        items.append({"label": f"Q{qnum}", "question": question, "text": answer})
    return items


def build_units(text: str, structure: dict[str, Any]) -> list[dict[str, Any]]:
    fmt = structure.get("transcript_format")
    if fmt in {"speaker_labeled", "mixed"}:
        turns = parse_speaker_turns(text)
        if turns:
            return [{"unit_type": "turn", "index": i + 1, **t} for i, t in enumerate(turns)]
    qas = parse_numbered_qa(text)
    if qas:
        return [{"unit_type": "qa", "index": i + 1, **x} for i, x in enumerate(qas)]
    # Fallback: whole transcript as one unit.
    return [{"unit_type": "raw", "index": 1, "label": "RAW", "text": text}]


def render_unit(unit: dict[str, Any]) -> str:
    if unit["unit_type"] == "qa":
        return f"{unit['label']}. {unit.get('question','')}\nANSWER: {unit['text']}"
    return f"{unit.get('label','')}: {unit['text']}"


def chunk_units(units: list[dict[str, Any]], max_chars: int = CHUNK_MAX_CHARS) -> list[list[dict[str, Any]]]:
    chunks: list[list[dict[str, Any]]] = []
    current: list[dict[str, Any]] = []
    current_chars = 0
    for unit in units:
        rendered = render_unit(unit)
        size = len(rendered)
        if current and current_chars + size > max_chars:
            chunks.append(current)
            overlap = current[-CHUNK_OVERLAP_TURNS:] if CHUNK_OVERLAP_TURNS > 0 else []
            current = list(overlap)
            current_chars = sum(len(render_unit(x)) for x in current)
        current.append(unit)
        current_chars += size
    if current:
        chunks.append(current)
    return chunks


## 9. Evidence coding with verbatim quotation verification

In [10]:

def normalized_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def verify_quote(quote: str, source_text: str) -> bool:
    if not quote.strip():
        return False
    return normalized_ws(quote) in normalized_ws(source_text)


def code_chunk(
    client: OpenAI,
    *,
    transcript_id: str,
    chunk_index: int,
    total_chunks: int,
    chunk_text: str,
    structure: dict[str, Any],
    expert_meta: ExpertMeta | None,
    guide_text: str,
) -> dict[str, Any]:
    codebook_text = "\n".join(f"{k}: {v}" for k, v in CODEBOOK.items())
    rq_text = "\n".join(f"{k}: {v}" for k, v in RESEARCH_QUESTIONS.items())

    cache_key = fingerprint(
        chunk_text,
        json.dumps(structure, ensure_ascii=False, sort_keys=True),
        codebook_text,
        rq_text,
    )
    cache_path = CACHE_DIR / safe_slug(transcript_id) / f"{cache_key[:20]}_CHUNK_{chunk_index:03d}.json"
    cached = load_cache(cache_path)
    if cached is not None:
        return cached

    meta_payload = None
    if expert_meta:
        meta_payload = {
            "name": expert_meta.name,
            "age": expert_meta.age,
            "description": expert_meta.description,
            "sector": expert_meta.sector,
        }

    instructions = COMMON_EVIDENCE_RULES + "\n\n" + f"""
RESEARCH QUESTIONS:\n{rq_text}\n\nFIXED CODEBOOK:\n{codebook_text}

Coding task:
- Code only substantive EXPERT material in this chunk.
- One evidence item should represent one analytically coherent claim or example.
- question_ids may contain Q1..Q19 or Q20+ when the relation is supported. Use an empty array if no guide item fits.
- primary_code must be the best fixed code; secondary_codes only if genuinely co-occurring.
- emergent_code should be a concise Ukrainian label when a meaningful idea is not captured by the fixed codebook; otherwise return an empty string.
- analytic_memo should explain why the excerpt matters for the article, not merely repeat it.
- Preserve negative cases: e.g., an expert who says technologies do NOT alter their artistic work is analytically important.
- Do not confuse personal social-media use with sector-level distribution or institutional platform use.
""".strip()

    prompt = json.dumps({
        "transcript_id": transcript_id,
        "chunk": f"{chunk_index}/{total_chunks}",
        "expert_metadata": meta_payload,
        "role_mapping": structure,
        "questionnaire": guide_text[:20_000],
        "transcript_chunk": chunk_text,
    }, ensure_ascii=False, indent=2)

    result = call_structured(
        client,
        model=CODING_MODEL,
        instructions=instructions,
        input_text=prompt,
        schema=CHUNK_SCHEMA,
        schema_name="coded_interview_chunk",
        max_output_tokens=6_000,
        stage=f"coding_chunk_{chunk_index}",
        transcript_id=transcript_id,
    )
    save_cache(cache_path, result)
    return result


def flatten_verified_evidence(
    transcript_id: str,
    expert_meta: ExpertMeta | None,
    structure: dict[str, Any],
    chunk_index: int,
    chunk_text: str,
    chunk_result: dict[str, Any],
) -> tuple[list[dict[str, Any]], int]:
    rows: list[dict[str, Any]] = []
    rejected = 0
    for item_no, item in enumerate(chunk_result.get("evidence", []), start=1):
        ok = verify_quote(item.get("quote", ""), chunk_text)
        if not ok:
            rejected += 1
            continue
        secondary = item.get("secondary_codes", [])
        mechanisms = item.get("mediatization_mechanisms", [])
        rows.append({
            "transcript_id": transcript_id,
            "expert_name": expert_meta.name if expert_meta else structure.get("respondent_name_from_text", ""),
            "age": expert_meta.age if expert_meta else None,
            "sector": resolve_sector(expert_meta, structure),
            "chunk_index": chunk_index,
            "evidence_no": item_no,
            "question_ids": " | ".join(item.get("question_ids", [])),
            "primary_code": item.get("primary_code", ""),
            "secondary_codes": " | ".join(secondary),
            "emergent_code": item.get("emergent_code", ""),
            "mediatization_mechanisms": " | ".join(mechanisms),
            "stance": item.get("stance", ""),
            "ai_relevance": item.get("ai_relevance", ""),
            "sector_specificity": item.get("sector_specificity", ""),
            "quote": item.get("quote", ""),
            "paraphrase": item.get("paraphrase", ""),
            "analytic_memo": item.get("analytic_memo", ""),
            "confidence": item.get("confidence", None),
            "quote_verified": True,
        })
    return rows, rejected


## 10. Within-interview synthesis

In [11]:

def synthesize_interview(
    client: OpenAI,
    *,
    transcript_id: str,
    expert_meta: ExpertMeta | None,
    structure: dict[str, Any],
    evidence_rows: list[dict[str, Any]],
    chunk_notes: list[dict[str, Any]],
) -> dict[str, Any]:
    payload = {
        "expert_metadata": None if expert_meta is None else expert_meta.__dict__,
        "structure": structure,
        "verified_evidence": evidence_rows,
        "chunk_notes": [
            {
                "chunk_summary": x.get("chunk_summary", ""),
                "contradictions_or_qualifications": x.get("contradictions_or_qualifications", []),
                "missing_or_uncertain_material": x.get("missing_or_uncertain_material", []),
            }
            for x in chunk_notes
        ],
    }
    payload_text = json.dumps(payload, ensure_ascii=False, indent=2)
    key = fingerprint(payload_text)
    cache_path = CACHE_DIR / safe_slug(transcript_id) / f"{key[:20]}_SUMMARY.json"
    cached = load_cache(cache_path)
    if cached is not None:
        return cached

    instructions = COMMON_EVIDENCE_RULES + "\n\n" + """
Synthesize ONE interview from verified coded evidence.
Do not introduce new quotations. Do not infer prevalence across other interviews.
Separate: (a) deep mediatization of professional practice; (b) AI transformation;
(c) professional identity/boundaries; (d) audience/visibility; (e) economy/platforms;
(f) transformation of artistic form/product.
Pay special attention to tensions and negative cases rather than forcing a single pro-technology narrative.
Possible propositions must be phrased as hypotheses/analytical propositions for later cross-case testing, not as proven general laws.
""".strip()

    result = call_structured(
        client,
        model=SYNTHESIS_MODEL,
        instructions=instructions,
        input_text=payload_text,
        schema=INTERVIEW_SUMMARY_SCHEMA,
        schema_name="interview_summary",
        max_output_tokens=5_500,
        stage="interview_synthesis",
        transcript_id=transcript_id,
    )
    save_cache(cache_path, result)
    return result


## 11. Aggregation: participant-by-code, sector-by-code, and negative cases

In [12]:

def build_aggregate_tables(evidence_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    if evidence_df.empty:
        return {
            "participant_code": pd.DataFrame(),
            "sector_code": pd.DataFrame(),
            "code_stance": pd.DataFrame(),
            "emergent": pd.DataFrame(),
        }

    base = evidence_df.copy()
    base["present"] = 1

    participant_code = pd.pivot_table(
        base,
        index=["transcript_id", "expert_name", "sector"],
        columns="primary_code",
        values="present",
        aggfunc="max",
        fill_value=0,
    ).reset_index()

    # Count distinct interviews containing a code, not the number of quotations.
    sector_code = (
        base[["sector", "primary_code", "transcript_id"]]
        .drop_duplicates()
        .groupby(["sector", "primary_code"], as_index=False)
        .agg(interviews_with_code=("transcript_id", "nunique"))
    )

    code_stance = (
        base[["primary_code", "stance", "transcript_id"]]
        .drop_duplicates()
        .groupby(["primary_code", "stance"], as_index=False)
        .agg(interviews=("transcript_id", "nunique"))
    )

    emergent = (
        base.loc[base["emergent_code"].fillna("").str.strip() != "", ["emergent_code", "transcript_id", "sector"]]
        .drop_duplicates()
        .groupby("emergent_code", as_index=False)
        .agg(
            interviews=("transcript_id", "nunique"),
            sectors=("sector", lambda x: " | ".join(sorted(set(map(str, x))))),
        )
        .sort_values(["interviews", "emergent_code"], ascending=[False, True])
    )

    return {
        "participant_code": participant_code,
        "sector_code": sector_code,
        "code_stance": code_stance,
        "emergent": emergent,
    }


def select_quote_bank(evidence_df: pd.DataFrame, per_code: int = 4) -> list[dict[str, Any]]:
    if evidence_df.empty:
        return []
    rows = []
    for code, group in evidence_df.sort_values(["confidence"], ascending=False).groupby("primary_code"):
        seen_interviews = set()
        for _, r in group.iterrows():
            if r["transcript_id"] in seen_interviews:
                continue
            seen_interviews.add(r["transcript_id"])
            rows.append({
                "primary_code": code,
                "transcript_id": r["transcript_id"],
                "expert_name": r["expert_name"],
                "sector": r["sector"],
                "quote": r["quote"],
                "analytic_memo": r["analytic_memo"],
            })
            if len(seen_interviews) >= per_code:
                break
    return rows


## 12. Cross-case synthesis for the academic article

In [13]:

def cross_case_synthesis(
    client: OpenAI,
    interview_rows: list[dict[str, Any]],
    interview_summaries: dict[str, dict[str, Any]],
    evidence_df: pd.DataFrame,
    aggregates: dict[str, pd.DataFrame],
    survey_axis_map: dict[str, list[str]],
) -> dict[str, Any]:
    # Full per-interview coding stays intact in Excel and cached JSON files.
    # Only the cross-case API request is compacted to fit the configured limit.
    def compact_value(value, max_chars):
        if isinstance(value, str):
            return value if len(value) <= max_chars else value[:max_chars].rsplit(" ", 1)[0] + " [truncated for synthesis]"
        if isinstance(value, list):
            return [compact_value(item, max_chars) for item in value]
        if isinstance(value, dict):
            return {key: compact_value(item, max_chars) for key, item in value.items()}
        return value

    def make_payload(per_code, summary_limit, memo_limit):
        quote_bank = select_quote_bank(evidence_df, per_code=per_code)
        trimmed_quotes = [
            {**row, "analytic_memo": compact_value(row.get("analytic_memo", ""), memo_limit)}
            for row in quote_bank
        ]
        slim_metadata = [
            {key: row.get(key) for key in (
                "transcript_id", "expert_name", "sector", "expert_description",
                "role_confidence", "quality_notes", "verified_evidence_items"
            )}
            for row in interview_rows
        ]
        return {
            "article_title": ARTICLE_TITLE,
            "research_questions": RESEARCH_QUESTIONS,
            "sample_metadata": slim_metadata,
            "interview_summaries": compact_value(interview_summaries, summary_limit),
            "participant_code_matrix": aggregates["participant_code"].to_dict("records"),
            "sector_code_counts": aggregates["sector_code"].to_dict("records"),
            "code_stance_counts": aggregates["code_stance"].to_dict("records"),
            "emergent_codes": aggregates["emergent"].to_dict("records"),
            "verified_quote_bank": trimmed_quotes,
            "survey_question_alignment": survey_axis_map,
        }

    # Prefer breadth across the interview sample and analytic categories.
    strategies = [(4, 2400, 900), (3, 1400, 600), (2, 900, 400),
                  (1, 600, 250), (1, 350, 120)]
    text_payload = None
    for per_code, summary_limit, memo_limit in strategies:
        payload = make_payload(per_code, summary_limit, memo_limit)
        candidate = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
        print(f"Cross-case synthesis payload: {len(candidate):,} / {MAX_SYNTHESIS_INPUT_CHARS:,} characters "
              f"(up to {per_code} quotations per code)")
        if len(candidate) <= MAX_SYNTHESIS_INPUT_CHARS:
            text_payload = candidate
            break
    if text_payload is None:
        raise ValueError(
            "Even the compact cross-case payload exceeds MAX_SYNTHESIS_INPUT_CHARS. "
            "Increase the limit or use a hierarchical sector-by-sector synthesis. "
            "Interview coding remains saved in the cache."
        )

    key = fingerprint(text_payload)
    cache_path = CACHE_DIR / f"{key[:20]}_CROSS_CASE.json"
    cached = load_cache(cache_path)
    if cached is not None:
        return cached

    instructions = COMMON_EVIDENCE_RULES + "\n\n" + """
Perform cross-case qualitative synthesis for an academic article.

Hard constraints:
- The unit of prevalence is an INTERVIEW, not a coded quote. Use supplied matrices/counts for any statements such as 'in N interviews'.
- Do not extrapolate to all Ukrainian artists or the general population.
- Distinguish common patterns from sector-specific patterns and negative cases.
- Distinguish deep mediatization mechanisms from AI-specific transformation.
- Do not treat 'uses digital tools' as equivalent to 'uses generative AI'.
- Preserve cases where respondents reject or minimize technological influence on the artistic core of their work.
- Use verified quotes only as illustration. Do not generate new quotations.
- In qualitative_results_draft, write a coherent journal-style RESULTS section in academic English, but do not insert invented theory citations.
- survey_integration_plan must specify which KII findings can be triangulated with existing survey questions, without pretending the survey has already been statistically analysed.
- claims_not_supported should explicitly list tempting but unsupported generalizations.
""".strip()

    result = call_structured(
        client,
        model=SYNTHESIS_MODEL,
        instructions=instructions,
        input_text=text_payload,
        schema=CROSS_CASE_SCHEMA,
        schema_name="cross_case_article_synthesis",
        max_output_tokens=9_000,
        stage="cross_case_synthesis",
        transcript_id="ALL_INTERVIEWS",
    )
    save_cache(cache_path, result)
    return result


## 13. Export to Excel and Markdown

In [14]:

def excel_safe(v: Any) -> Any:
    if isinstance(v, (list, tuple, set)):
        v = " | ".join(map(str, v))
    elif isinstance(v, dict):
        v = json.dumps(v, ensure_ascii=False)
    if isinstance(v, str) and len(v) > 32_000:
        return v[:31_500] + " [truncated for Excel]"
    return v


def style_ws(ws, widths: dict[str, int] | None = None) -> None:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    fill = PatternFill("solid", fgColor="1F4E78")
    for cell in ws[1]:
        cell.fill = fill
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    for idx, col in enumerate(ws.columns, start=1):
        header = str(col[0].value or "")
        if widths and header in widths:
            width = widths[header]
        else:
            sample = [str(c.value or "") for c in list(col)[:80]]
            width = min(max(max((len(x) for x in sample), default=8) + 2, 10), 45)
        ws.column_dimensions[get_column_letter(idx)].width = width


def export_results(
    interview_rows: list[dict[str, Any]],
    summaries: dict[str, dict[str, Any]],
    evidence_df: pd.DataFrame,
    aggregates: dict[str, pd.DataFrame],
    cross_case: dict[str, Any],
    survey_axis_map: dict[str, list[str]],
    run_log: list[dict[str, Any]],
) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    summaries_rows = []
    for tid, summary in summaries.items():
        row = {"transcript_id": tid, **summary}
        summaries_rows.append({k: excel_safe(v) for k, v in row.items()})

    codebook_df = pd.DataFrame([
        {"code": code, "definition": definition}
        for code, definition in CODEBOOK.items()
    ])

    survey_map_df = pd.DataFrame([
        {"integration_axis": axis, "survey_questions": " | ".join(qs)}
        for axis, qs in survey_axis_map.items()
    ])

    cross_df = pd.DataFrame([
        {"field": k, "value": excel_safe(v)} for k, v in cross_case.items()
    ])

    dfs = {
        "Interviews": pd.DataFrame(interview_rows),
        "Interview_summaries": pd.DataFrame(summaries_rows),
        "Coded_evidence": evidence_df,
        "Participant_code": aggregates["participant_code"],
        "Sector_code": aggregates["sector_code"],
        "Code_stance": aggregates["code_stance"],
        "Emergent_codes": aggregates["emergent"],
        "Codebook": codebook_df,
        "Survey_alignment": survey_map_df,
        "Cross_case": cross_df,
        "Run_log": pd.DataFrame(run_log + API_CALL_LOG),
    }

    with pd.ExcelWriter(RESULT_XLSX, engine="openpyxl") as writer:
        for sheet, df in dfs.items():
            safe_df = df.copy()
            for col in safe_df.columns:
                safe_df[col] = safe_df[col].map(excel_safe)
            safe_df.to_excel(writer, sheet_name=sheet[:31], index=False)

        wb = writer.book
        for ws in wb.worksheets:
            widths = {"quote": 70, "analytic_memo": 70, "analytical_summary": 80, "value": 100}
            style_ws(ws, widths=widths)

    lines = [
        f"# {ARTICLE_TITLE}",
        "",
        "## Central argument",
        cross_case.get("central_argument", ""),
        "",
        "## Draft qualitative findings",
        cross_case.get("qualitative_results_draft", ""),
        "",
        "## Deep mediatization",
    ]
    lines += [f"- {x}" for x in cross_case.get("deep_mediatization_findings", [])]
    lines += ["", "## AI transformation"]
    lines += [f"- {x}" for x in cross_case.get("ai_transformation_findings", [])]
    lines += ["", "## Sectoral differences"]
    lines += [f"- {x}" for x in cross_case.get("sector_differences", [])]
    lines += ["", "## Mass-survey integration plan"]
    lines += [f"- {x}" for x in cross_case.get("survey_integration_plan", [])]
    lines += ["", "## Claims not supported by the current evidence"]
    lines += [f"- {x}" for x in cross_case.get("claims_not_supported", [])]
    ARTICLE_MEMO_MD.write_text("\n".join(lines), encoding="utf-8")


## 14. Preview before running paid API requests

In [ ]:

experts = load_expert_directory(KII_LIST_PATH)
transcripts = discover_transcripts(INPUT_DIR) if INPUT_DIR.exists() else []
survey_headers = load_survey_headers()
survey_axis_map = map_survey_axes(survey_headers)

preview_rows = []
for path in transcripts:
    text, enc = read_text_file(path)
    expert, score = match_expert(path.name, experts)
    preview_rows.append({
        "file": path.name,
        "characters": len(text),
        "encoding": enc,
        "matched_expert": expert.name if expert else "",
        "sector": expert.sector if expert else "unknown",
        "filename_match_score": round(score, 3),
    })

print(f"Transcripts found: {len(transcripts)}")
print(f"Experts in directory: {len(experts)}")
print(f"Survey questions available for later triangulation: {len(survey_headers)}")
if preview_rows:
    display(pd.DataFrame(preview_rows))
else:
    print(f"Set INPUT_DIR to the folder with TXT transcripts. Current: {INPUT_DIR}")

print("\nSurvey/KII alignment axes:")
display(pd.DataFrame([
    {"axis": k, "questions_found": len(v), "questions": " | ".join(v)}
    for k, v in survey_axis_map.items()
]))


## 15. Execute the full analysis

In [ ]:

def run_pipeline() -> tuple[Path, Path]:
    if OpenAI is None:
        raise ImportError("openai package is not installed")
    if not INPUT_DIR.exists():
        raise FileNotFoundError(INPUT_DIR)

    transcript_paths = discover_transcripts(INPUT_DIR)
    if not transcript_paths:
        raise ValueError(f"No transcript TXT files found in {INPUT_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    expert_directory = load_expert_directory(KII_LIST_PATH)
    guide_text = load_guide_text()
    survey_headers_local = load_survey_headers()
    survey_axis_map_local = map_survey_axes(survey_headers_local)

    client = OpenAI(api_key=load_api_key(), timeout=REQUEST_TIMEOUT_SECONDS)

    interview_rows: list[dict[str, Any]] = []
    all_evidence_rows: list[dict[str, Any]] = []
    interview_summaries: dict[str, dict[str, Any]] = {}
    run_log: list[dict[str, Any]] = []

    for idx, path in enumerate(transcript_paths, start=1):
        transcript_id = path.stem
        print(f"\n[{idx}/{len(transcript_paths)}] {path.name}")
        started = time.time()
        try:
            raw_text, encoding = read_text_file(path)
            text = clean_text(raw_text)
            expert_meta, match_score = match_expert(path.name, expert_directory)

            structure = detect_structure(
                client,
                transcript_id,
                text,
                expert_meta,
                match_score,
                guide_text,
            )
            print(
                f"  format={structure['transcript_format']} | "
                f"interviewer={structure['interviewer_label']} | expert={structure['expert_label']} | "
                f"role_confidence={structure['role_confidence']}"
            )

            units = build_units(text, structure)
            chunks = chunk_units(units)
            print(f"  units={len(units)} | chunks={len(chunks)}")

            chunk_results = []
            verified_rows: list[dict[str, Any]] = []
            rejected_total = 0

            for cidx, chunk in enumerate(chunks, start=1):
                chunk_text = "\n\n".join(render_unit(x) for x in chunk)
                result = code_chunk(
                    client,
                    transcript_id=transcript_id,
                    chunk_index=cidx,
                    total_chunks=len(chunks),
                    chunk_text=chunk_text,
                    structure=structure,
                    expert_meta=expert_meta,
                    guide_text=guide_text,
                )
                chunk_results.append(result)
                rows, rejected = flatten_verified_evidence(
                    transcript_id,
                    expert_meta,
                    structure,
                    cidx,
                    chunk_text,
                    result,
                )
                verified_rows.extend(rows)
                rejected_total += rejected

            summary = synthesize_interview(
                client,
                transcript_id=transcript_id,
                expert_meta=expert_meta,
                structure=structure,
                evidence_rows=verified_rows,
                chunk_notes=chunk_results,
            )

            all_evidence_rows.extend(verified_rows)
            interview_summaries[transcript_id] = summary

            row = {
                "transcript_id": transcript_id,
                "file": path.name,
                "expert_name": expert_meta.name if expert_meta else structure.get("respondent_name_from_text", ""),
                "age": expert_meta.age if expert_meta else None,
                "expert_description": expert_meta.description if expert_meta else "",
                "sector": resolve_sector(expert_meta, structure),
                "filename_match_score": round(match_score, 3),
                "transcript_format": structure.get("transcript_format", ""),
                "interviewer_label": structure.get("interviewer_label", ""),
                "expert_label": structure.get("expert_label", ""),
                "role_confidence": structure.get("role_confidence", None),
                "role_rationale": structure.get("role_rationale", ""),
                "primary_language": structure.get("primary_language", ""),
                "quality_notes": " | ".join(structure.get("quality_notes", [])),
                "characters": len(text),
                "encoding": encoding,
                "units": len(units),
                "chunks": len(chunks),
                "verified_evidence_items": len(verified_rows),
                "rejected_unverified_quotes": rejected_total,
            }
            interview_rows.append(row)
            run_log.append({
                "transcript_id": transcript_id,
                "stage": "transcript_complete",
                "status": "ok",
                "elapsed_seconds": round(time.time() - started, 2),
                "error": "",
            })
            print(f"  verified evidence={len(verified_rows)} | rejected quotes={rejected_total}")

        except Exception as exc:
            run_log.append({
                "transcript_id": transcript_id,
                "stage": "transcript_complete",
                "status": "failed",
                "elapsed_seconds": round(time.time() - started, 2),
                "error": str(exc),
            })
            print(f"  FAILED: {exc}")

    evidence_df = pd.DataFrame(all_evidence_rows)
    aggregates = build_aggregate_tables(evidence_df)

    successful_interviews = [r for r in interview_rows if r["transcript_id"] in interview_summaries]
    if not successful_interviews:
        raise RuntimeError("No interviews completed successfully")

    cross_case = cross_case_synthesis(
        client,
        successful_interviews,
        interview_summaries,
        evidence_df,
        aggregates,
        survey_axis_map_local,
    )

    export_results(
        successful_interviews,
        interview_summaries,
        evidence_df,
        aggregates,
        cross_case,
        survey_axis_map_local,
        run_log,
    )

    print(f"\nExcel: {RESULT_XLSX}")
    print(f"Article memo: {ARTICLE_MEMO_MD}")
    return RESULT_XLSX, ARTICLE_MEMO_MD


if RUN_ANALYSIS:
    result_xlsx, result_md = run_pipeline()
else:
    print("RUN_ANALYSIS=False. Review the preview and file paths, then change to True.")
